# Notebook 03 — Conformal Prediction Calibration

Apply split conformal prediction to XGBoost across the 3 splits and document coverage breakdown.

In [ ]:
import sys; sys.path.insert(0, '..')import pandas as pdimport numpy as npimport jsonfrom src.splits import build_splitsfrom src.models import train_xgb_with_valfrom src.conformal import evaluate_conformal_coveragedf = pd.read_parquet('../data/chf_clean.parquet')FEATURES = ['Tube Diameter', 'Heated Length', 'Pressure', 'Mass Flux', 'Outlet Quality']X = df[FEATURES].values.astype(np.float32)y = df['CHF'].values.astype(np.float32)splits = build_splits(X, y, df, seed=42)results = {}for split_name, (X_tr, y_tr, X_cal, y_cal, X_te, y_te) in splits.items():    m = train_xgb_with_val(X_tr, y_tr, X_cal, y_cal, seed=42)    cp = evaluate_conformal_coverage(m, X_cal, y_cal, X_te, y_te, alpha=0.10)    results[split_name] = cp    print(f'{split_name}: coverage={cp["empirical_coverage"]*100:.1f}%, q_hat={cp["q_hat"]:.1f}')with open('../results/conformal_reproduced.json', 'w') as f:    json.dump(results, f, indent=2)